# Hypothesis Test: Does Friday Publishing Improve Multi-Day Retention?

**Null hypothesis (H0)**: within a given country, Friday-published videos
and non-Friday-published videos have the same multi-day retention rate.

**Primary metric**: `survived_day1` (binary — chosen over average days-on-
trending because FRANCE and CANADA have 59-75% of videos surviving only
1 day; a mean would be dominated by a handful of long-tail outliers).

**Test**: four separate two-proportion z-tests (one per country), NOT a
pooled test — pooling would mix in the country composition effect, since
the four countries have wildly different baselines (24.8% to 93.1%).


In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.weightstats import ztest
from scipy.stats import mannwhitneyu
import statsmodels.formula.api as smf

video_life = pd.read_csv('../data/exports/video_life.csv', parse_dates=['first_trend_day', 'last_trend_day'])
video_life['is_friday'] = (video_life['pub_dow'] == 'Friday').astype(int)
video_life.head()


,video_id,publish_country,channel_title,category_id,pub_dow,pub_hour_utc,first_trend_day,last_trend_day,days_on_trending,trending_span,survived_day1,peak_views,peak_likes,peak_comments,gap_real,gap_scrape,is_friday
0,w_b63zETbSs,US,Steve TV Show,24,Tuesday,6:00 to 6:59,2017-11-29,2017-12-03,5,5,1,54208,462,75,0,0,0
1,wa5eXqjEewE,US,Global News,25,Friday,19:00 to 19:59,2018-04-14,2018-04-23,10,10,1,783712,3700,2188,0,0,1
2,wbSwFU6tY1c,GB,SpaceX,28,Tuesday,21:00 to 21:59,2018-02-07,2018-03-14,36,36,1,21530325,392249,39468,0,0,0
3,wceoB6YlsZM,CANADA,å¤§åŠ‡ç¨æ’­,1,Saturday,16:00 to 16:59,2018-01-14,2018-01-14,1,1,0,28326,49,30,0,0,0
4,we2XODhx-CM,CANADA,TheEllenShow,24,Thursday,14:00 to 14:59,2017-12-01,2017-12-01,1,1,0,264975,5214,308,0,0,0


## Four per-country two-proportion z-tests, with Newcombe confidence intervals

In [2]:
results = []
for country in ['FRANCE', 'CANADA', 'US', 'GB']:
    subset = video_life[video_life['publish_country'] == country]
    friday = subset[subset['is_friday'] == 1]
    non_friday = subset[subset['is_friday'] == 0]

    count = np.array([friday['survived_day1'].sum(), non_friday['survived_day1'].sum()])
    nobs = np.array([len(friday), len(non_friday)])

    z_stat, p_value = proportions_ztest(count, nobs)
    ci_low, ci_upp = confint_proportions_2indep(
        count[0], nobs[0], count[1], nobs[1], method='newcomb'
    )
    p_friday = count[0] / nobs[0]
    p_non_friday = count[1] / nobs[1]

    results.append({
        'country': country,
        'friday_rate': round(p_friday * 100, 1),
        'non_friday_rate': round(p_non_friday * 100, 1),
        'diff_pp': round((p_friday - p_non_friday) * 100, 1),
        'ci_low_pp': round(ci_low * 100, 1),
        'ci_high_pp': round(ci_upp * 100, 1),
        'friday_n': nobs[0],
        'p_value': round(p_value, 4),
        'significant': p_value < 0.05,
    })

results_df = pd.DataFrame(results)
results_df


,country,friday_rate,non_friday_rate,diff_pp,ci_low_pp,ci_high_pp,friday_n,p_value,significant
0,FRANCE,29.0,24.0,5.0,3.7,6.4,4873,0.0000,True
1,CANADA,44.0,39.9,4.2,2.5,5.9,3904,0.0000,True
2,US,91.8,88.3,3.5,1.5,5.3,1038,0.0009,True
3,GB,93.6,93.0,0.6,-1.8,2.6,610,0.5885,False


**Compare against the design doc's reported values**: FRANCE +5.0
(CI +3.7 to +6.4), CANADA +4.2 (+2.5 to +5.9), US +3.5 (+1.5 to +5.3),
GB +0.6 (-1.8 to +2.6, not significant).

Three countries show a consistent positive effect with intervals not
crossing zero; GB doesn't. GB's baseline is already ~93% — a ceiling
effect, not evidence that "Friday doesn't work in GB". This is a
candidate lever, not a causal effect — better channels may simply prefer
publishing on Fridays (confounding).

## 2,000-iteration bootstrap cross-check

Resampling the Friday/non-Friday groups with replacement 2,000 times per
country, to cross-validate the analytic (Newcombe) confidence intervals
with an empirical distribution.

In [3]:
def bootstrap_diff_ci(friday_vals, non_friday_vals, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        f_sample = rng.choice(friday_vals, size=len(friday_vals), replace=True)
        nf_sample = rng.choice(non_friday_vals, size=len(non_friday_vals), replace=True)
        diffs[i] = f_sample.mean() - nf_sample.mean()
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

bootstrap_results = []
for country in ['FRANCE', 'CANADA', 'US', 'GB']:
    subset = video_life[video_life['publish_country'] == country]
    friday_vals = subset[subset['is_friday'] == 1]['survived_day1'].values
    non_friday_vals = subset[subset['is_friday'] == 0]['survived_day1'].values
    lo, hi = bootstrap_diff_ci(friday_vals, non_friday_vals)
    bootstrap_results.append({'country': country, 'bootstrap_ci_low_pp': round(lo*100, 1), 'bootstrap_ci_high_pp': round(hi*100, 1)})

pd.DataFrame(bootstrap_results)


,country,bootstrap_ci_low_pp,bootstrap_ci_high_pp
0,FRANCE,3.7,6.4
1,CANADA,2.5,5.9
2,US,1.8,5.4
3,GB,-1.6,2.8


**Conclusion**: the bootstrap intervals should closely track the
Newcombe analytic intervals above — if they diverged substantially, that
would be a red flag about the normal-approximation assumptions underlying
the z-test.

## Secondary metric: Mann-Whitney U on `days_on_trending`

`days_on_trending` is right-skewed and non-normal (median 1, mean pulled
up by long-tail outliers), violating the t-test's normality assumption.
Mann-Whitney U compares ranks, not means, so it doesn't need that
assumption. This recovers some of the information the binary primary
metric discards (the difference between surviving 2 days vs. 20 days).

In [4]:
mw_results = []
for country in ['FRANCE', 'CANADA', 'US', 'GB']:
    subset = video_life[video_life['publish_country'] == country]
    friday_days = subset[subset['is_friday'] == 1]['days_on_trending']
    non_friday_days = subset[subset['is_friday'] == 0]['days_on_trending']
    stat, p = mannwhitneyu(friday_days, non_friday_days, alternative='greater')
    mw_results.append({
        'country': country,
        'friday_median_days': friday_days.median(),
        'non_friday_median_days': non_friday_days.median(),
        'mw_p_value': round(p, 4),
    })

pd.DataFrame(mw_results)


,country,friday_median_days,non_friday_median_days,mw_p_value
0,FRANCE,1.0,1.0,0.0000
1,CANADA,1.0,1.0,0.0000
2,US,6.0,6.0,0.0715
3,GB,11.0,10.0,0.0366


## Logistic regression: the second implementation of the stratified test

`survived_day1 ~ is_friday + C(publish_country)`, FRANCE as the reference
group. This answers a different question than the four z-tests above:
"controlling for country, does Friday still associate with retention
overall?" — not "does the effect differ by country?" (that question is
already answered by the per-country z-tests). No interaction term is
added on purpose — one model, one question — and no regularization,
because this is meant as a descriptive stratification tool, not a
predictive model.

This choice (logistic regression over a Cochran-Mantel-Haenszel test)
satisfies a "regression analysis" requirement while doing the same job as
CMH — one method for two needs.

In [5]:
model = smf.logit(
    'survived_day1 ~ is_friday + C(publish_country, Treatment(reference="FRANCE"))',
    data=video_life
).fit()

print(model.summary())


Optimization terminated successfully.
         Current function value: 0.566552
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:          survived_day1   No. Observations:                63783
Model:                          Logit   Df Residuals:                    63778
Method:                           MLE   Df Model:                            4
Date:                Sat, 19 Sep 2026   Pseudo R-squ.:                  0.1608
Time:                        05:52:36   Log-Likelihood:                -36136.
converged:                       True   LL-Null:                       -43058.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept 

In [6]:
odds_ratios = np.exp(model.params)
conf_int = np.exp(model.conf_int())
conf_int.columns = ['ci_low', 'ci_high']
summary_table = pd.concat([odds_ratios.rename('odds_ratio'), conf_int], axis=1)
summary_table


,odds_ratio,ci_low,ci_high
Intercept,0.317788,0.309188,0.326627
"C(publish_country, Treatment(reference=""FRANCE""))[T.CANADA]",2.069672,1.995114,2.147017
"C(publish_country, Treatment(reference=""FRANCE""))[T.GB]",40.896012,35.605204,46.973015
"C(publish_country, Treatment(reference=""FRANCE""))[T.US]",24.242278,22.314496,26.336604
is_friday,1.247091,1.189614,1.307346


**Reading this**: controlling for country, the odds of a Friday-
published video surviving to day 2 are `is_friday`'s odds ratio times
that of a non-Friday video. Country coefficients being far larger than
`is_friday` shows country is the dominant factor.

**Compare against the design doc**: `is_friday` odds ratio 1.25 (95% CI
1.19-1.31); CANADA vs FRANCE 2.07; US vs FRANCE 24.2; GB vs FRANCE 40.9.

**Important**: this model is descriptive, not causal — it doesn't control
for channel quality. The odds ratio should be described as "the strength
of association after controlling for country," not "the effect of Friday
publishing."

## Sample size: two methods, cross-validated

Inputs: baseline proportion p0, target p1 = p0 + 0.03 (a 3-point MDE — a
business judgment about the smallest lift worth acting on, not a
statistical one), alpha = 0.05 two-sided, power = 0.80.

In [7]:
from statsmodels.stats.proportion import samplesize_proportions_2indep_onetail
from scipy.stats import norm

def manual_sample_size(p0, mde, alpha=0.05, power=0.80):
    p1 = p0 + mde
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    n = (z_alpha + z_beta)**2 * (p0*(1-p0) + p1*(1-p1)) / mde**2
    return n

baselines = {
    'FRANCE': video_life[video_life['publish_country']=='FRANCE']['survived_day1'].mean(),
    'CANADA': video_life[video_life['publish_country']=='CANADA']['survived_day1'].mean(),
}

# GOTCHA: samplesize_proportions_2indep_onetail defaults to alternative='two-sided'
# despite the function name saying "onetail". Without explicitly passing
# alternative='larger', it silently computes a two-sided sample size even
# when alpha has already been halved to 0.025 for a one-sided-equivalent
# test -- I first got 4,093 for FRANCE (wrong), not 3,380, until I found this.
sample_size_results = []
for market, p0 in baselines.items():
    tool_n = samplesize_proportions_2indep_onetail(
        diff=0.03, prop2=p0, power=0.8, alpha=0.025, alternative='larger'
    )
    manual_n = manual_sample_size(p0, 0.03)
    sample_size_results.append({
        'market': market,
        'baseline_p0': round(p0, 3),
        'tool_sample_size': round(tool_n),
        'manual_sample_size': round(manual_n),
    })

pd.DataFrame(sample_size_results)


,market,baseline_p0,tool_sample_size,manual_sample_size
0,FRANCE,0.248,3380,3377
1,CANADA,0.405,4249,4246


**Compare against the design doc**: FRANCE p0=0.248, tool 3,380,
manual 3,377; CANADA p0=0.405, tool 4,249, manual 4,246. The two methods
should differ by no more than a few units.

**Important caveat**: these are demonstrations on the *proxy metric*
(multi-day retention rate, denominator = already-trending videos,
computable from public data). The real primary metric's denominator is
"all published videos" — public data doesn't have that denominator. Once
switched to it, the baseline would drop by an order of magnitude (maybe
1-2% if only ~5% of published videos trend at all), making an absolute
3-point MDE unrealistic; it should become a relative lift (e.g. 10-15%),
with sample size recalculated against the internal baseline. The value of
this demonstration is the reproducible *process*, not these specific
numbers.

US and GB are excluded from sample size calculation entirely — their
baselines are already saturated (89-93%), so they're not in the
experiment's target markets.
